# OpenAI claim-coding pipeline

This notebook uses the OpenAI Responses API with `gpt-5-nano` to code an article for the **author's endorsed claim**.
It is explicitly **not** trying to summarize the whole article or pick generally salient sentences.

Expected outputs:
- `central_claim_summary`: one generated sentence stating the article's central claim
- `has_clear_central_thesis`: true/false
- `thesis_sentence_id`: exactly one sentence ID
- `support_sentence_ids`: up to 2 sentence IDs that support the same thesis
- `additional_claim_candidate_ids`: up to 3 extra claim-bearing sentence IDs when the article has multiple important claims or a diffuse thesis


In [29]:
import json
import os
from pathlib import Path
import re
import sys
import time
from textwrap import shorten

import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI

PROJECT_ROOT = next(
    (path for path in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (path / 'backend').exists()),
    Path.cwd().resolve(),
)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from backend.data_import import load_and_clean_guardian_years

load_dotenv(PROJECT_ROOT / '.env')

OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')

client = OpenAI(api_key=OPENAI_API_KEY)
MODEL = 'gpt-5-nano'
MODEL_SNAPSHOT = 'gpt-5-nano-2025-08-07'
REASONING_EFFORT = 'minimal'
MAX_OUTPUT_TOKENS = 250

In [3]:
def normalize_whitespace(text):
    return re.sub(r'\s+', ' ', str(text)).strip()


def split_into_sentences(text):
    normalized = normalize_whitespace(text)
    if not normalized:
        return []

    parts = re.split(r'(?<=[.!?])\s+', normalized)
    return [part.strip() for part in parts if part.strip()]


def sentence_table_from_text(article_text, prefix='s'):
    sentences = split_into_sentences(article_text)
    return pd.DataFrame(
        {
            'sentence_id': [f'{prefix}{idx}' for idx in range(len(sentences))],
            'sentence': sentences,
        }
    )


def select_article(
    articles=None,
    article_id=None,
    title_query=None,
    article_index=0,
    manual_text=None,
    manual_title='Manual article',
):
    if manual_text:
        article = pd.Series(
            {
                'id': 'manual',
                'title': manual_title,
                'summary': '',
                'date': pd.NaT,
                'author_display': '',
                'body_text': manual_text,
            }
        )
        return pd.DataFrame([article.to_dict()]), article

    if articles is None or articles.empty:
        raise ValueError('Provide a non-empty articles DataFrame or a MANUAL_ARTICLE_TEXT value.')

    matches = articles.copy()
    if article_id:
        matches = matches.loc[matches['id'].astype('string').str.strip() == str(article_id).strip()]
    elif title_query:
        matches = matches.loc[matches['title'].astype('string').str.contains(title_query, case=False, na=False)]

    matches = matches.reset_index(drop=True)
    if matches.empty:
        raise ValueError('No matching article found. Try a different ARTICLE_ID or TITLE_QUERY.')
    if not 0 <= int(article_index) < len(matches):
        raise IndexError(f'article_index {article_index} is out of range for {len(matches)} matching articles.')

    return matches, matches.iloc[int(article_index)]


def preview_article(article_row, body_width=1400):
    date_value = article_row.get('date', '')
    if pd.notna(date_value):
        try:
            date_value = pd.to_datetime(date_value).strftime('%Y-%m-%d')
        except Exception:
            date_value = str(date_value)

    display(Markdown(f"## {article_row.get('title', 'Untitled article')}"))
    print(f"id: {article_row.get('id', '')}")
    print(f"date: {date_value}")
    print(f"author(s): {article_row.get('author_display', '')}")
    print()
    print('Body preview:')
    print(shorten(normalize_whitespace(article_row.get('body_text', '')), width=body_width, placeholder=' ...'))


def sentence_lookup(sentence_df):
    return dict(zip(sentence_df['sentence_id'], sentence_df['sentence']))


In [ ]:
CLAIM_CODING_INSTRUCTIONS = """
You are annotating argument structure in opinion and analysis articles.

Your task is NOT generic summarization, NOT general salience detection, and NOT evidence extraction for an external query.
Your task is to identify the sentences that best capture the AUTHOR'S OWN ENDORSED ARGUMENT.

What to extract:
- the dominant central thesis of the article
- up to 2 major support sentences for that same thesis
- up to 3 secondary claim-bearing sentences if the article also advances other important endorsed claims

Key definitions:
- A thesis sentence is the single sentence that best states the author's main endorsed position.
- A support sentence states a major general reason, justification, principle, benefit, harm, or causal consequence supporting that thesis.
- A claim-bearing sentence expresses a debatable proposition, judgment, recommendation, normative position, or important causal argument relevant to the author's position.
- A secondary claim is an additional endorsed claim-bearing proposition that is important in the article but is not simply the same thesis sentence and is not merely an example or implementation detail.

What does NOT count as a good support or secondary claim unless it clearly states a broader proposition:
- concrete examples
- imagined scenarios
- case instances
- anecdotes
- operational or implementation details
- quoted speech
- opponent views
- background or scene-setting
- purely descriptive or factual context

Requirements for `central_claim_summary`:
- It must be exactly one sentence.
- It must state the article's central claim as a standalone proposition.
- Do NOT use attributional framing such as:
  - "the author argues"
  - "the article says"
  - "the writer believes"
  - "this piece claims"
- State the claim directly.
- Preserve the original polarity and modality where possible, such as "should", "must", "should not", "cannot", "is", or "is not".

Selection principles:
1. Prioritize the author's endorsed position, not merely a position mentioned in the article.
2. Prefer claim-bearing and reason-bearing sentences over generally important or topic-relevant sentences.
3. Prefer generalizable argumentative propositions over specific illustrations or applications.
4. Support sentences must support the same dominant thesis.
5. Support sentences should add distinct reasons rather than repeating the same point.
6. Secondary claims must be genuine endorsed claim-bearing propositions, not examples, restatements, or minor elaborations.
7. Choose only from the provided sentence IDs.
8. Return exactly one `thesis_sentence_id`.
9. If the article has no sharply stated thesis, still return the best thesis approximation and set `has_clear_central_thesis` to false.
10. Return 0 to 2 `support_sentence_ids`. Do not force supports if no good support sentences exist.
11. Return 0 to 3 `secondary_claim_ids`. Do not force secondary claims if none are strong enough.
12. Do not repeat the thesis ID in the support or secondary lists.
13. Do not repeat any sentence ID across fields.
14. If a sentence is mainly an example, scenario, anecdote, or implementation detail, do not select it unless it also clearly states a broader claim.

Tie-breaking guidance:
- Prefer sentences that directly state a position over sentences that merely imply it.
- Prefer sentences that are reusable as standalone propositions.
- Prefer sentences that express the article's organizing argument over sentences that only elaborate one branch of it.
- Prefer broader supporting reasons over narrow logistical details.
"""

CLAIM_CODING_SCHEMA = {
    'type': 'object',
    'additionalProperties': False,
    'properties': {
        'central_claim_summary': {
            'type': 'string',
            'description': 'Exactly one sentence stating the article’s central endorsed claim as a standalone proposition, without attributional framing.',
        },
        'has_clear_central_thesis': {
            'type': 'boolean',
            'description': 'Whether the article has a clear single dominant thesis rather than a diffuse or multi-claim structure.',
        },
        'thesis_sentence_id': {
            'type': 'string',
            'pattern': '^s\\d+$',
            'description': 'The single sentence ID that best states the author’s main endorsed position.',
        },
        'support_sentence_ids': {
            'type': 'array',
            'items': {'type': 'string', 'pattern': '^s\\d+$'},
            'maxItems': 2,
            'description': 'Up to 2 sentence IDs that provide distinct major support for the same dominant thesis. These should be general argumentative reasons, not merely examples or implementation details.',
        },
        'secondary_claim_ids': {
            'type': 'array',
            'items': {'type': 'string', 'pattern': '^s\\d+$'},
            'maxItems': 3,
            'description': 'Up to 3 additional endorsed claim-bearing sentence IDs that express important secondary or parallel claims, excluding the thesis, supports, examples, and mere restatements.',
        },
    },
    'required': [
        'central_claim_summary',
        'has_clear_central_thesis',
        'thesis_sentence_id',
        'support_sentence_ids',
        'secondary_claim_ids',
    ],
}

In [ ]:
def build_claim_coding_input(sentence_df, article_title=None):
    lines = []
    if article_title:
        lines.append(f'ARTICLE TITLE: {article_title}')
        lines.append('')

    lines.append('SENTENCES:')
    for row in sentence_df.itertuples(index=False):
        lines.append(f'{row.sentence_id}: {row.sentence}')

    lines.append('')
    lines.append('Return only the JSON object that matches the schema.')
    return '\n'.join(lines)


def extract_output_text(response):
    text = getattr(response, 'output_text', None)
    if text:
        return text

    chunks = []
    for item in getattr(response, 'output', []):
        if getattr(item, 'type', None) != 'message':
            continue
        for content in getattr(item, 'content', []):
            if getattr(content, 'type', None) == 'output_text':
                chunks.append(content.text)
    return ''.join(chunks)


def dedupe_preserve_order(values):
    deduped = []
    seen = set()
    for value in values:
        if value in seen:
            continue
        deduped.append(value)
        seen.add(value)
    return deduped


def usage_to_dict(response):
    usage = getattr(response, 'usage', None)
    if usage is None:
        return None

    output_details = getattr(usage, 'output_tokens_details', None)
    return {
        'input_tokens': getattr(usage, 'input_tokens', None),
        'output_tokens': getattr(usage, 'output_tokens', None),
        'total_tokens': getattr(usage, 'total_tokens', None),
        'reasoning_tokens': getattr(output_details, 'reasoning_tokens', None) if output_details else None,
    }


def validate_claim_coding_output(payload, sentence_df, response=None):
    valid_ids = set(sentence_df['sentence_id'])

    thesis_id = payload['thesis_sentence_id']
    support_ids = dedupe_preserve_order(payload.get('support_sentence_ids', []))[:2]
    additional_ids = dedupe_preserve_order(payload.get('secondary_claim_ids', []))[:3]

    if thesis_id not in valid_ids:
        raise ValueError(f'thesis_sentence_id {thesis_id!r} is not one of the provided sentence IDs.')

    invalid_support = [sid for sid in support_ids if sid not in valid_ids]
    invalid_additional = [sid for sid in additional_ids if sid not in valid_ids]
    if invalid_support:
        raise ValueError(f'Invalid support sentence IDs returned: {invalid_support}')
    if invalid_additional:
        raise ValueError(f'Invalid additional claim candidate IDs returned: {invalid_additional}')

    if thesis_id in support_ids or thesis_id in additional_ids:
        raise ValueError('The thesis sentence ID must not be repeated in support or additional lists.')

    overlap = set(support_ids) & set(additional_ids)
    if overlap:
        raise ValueError(f'Support and additional candidate IDs overlap: {sorted(overlap)}')

    cleaned = {
        'central_claim_summary': normalize_whitespace(payload['central_claim_summary']),
        'has_clear_central_thesis': bool(payload['has_clear_central_thesis']),
        'thesis_sentence_id': thesis_id,
        'support_sentence_ids': support_ids,
        'secondary_claim_ids': additional_ids,
        'response_id': getattr(response, 'id', None) if response is not None else None,
        'model': getattr(response, 'model', None) if response is not None else None,
        'usage': usage_to_dict(response) if response is not None else None,
    }
    return cleaned


def call_claim_coder(
    sentence_df,
    article_title=None,
    model=MODEL,
    reasoning_effort=REASONING_EFFORT,
    max_output_tokens=MAX_OUTPUT_TOKENS,
):
    if sentence_df.empty:
        raise ValueError('sentence_df is empty; there are no sentences to code.')

    response = client.responses.create(
        model=model,
        instructions=CLAIM_CODING_INSTRUCTIONS,
        input=build_claim_coding_input(sentence_df, article_title=article_title),
        reasoning={'effort': reasoning_effort},
        max_output_tokens=max_output_tokens,
        store=False,
        text={
            'verbosity': 'low',
            'format': {
                'type': 'json_schema',
                'name': 'article_claim_coding',
                'description': 'Structured coding of an article\'s dominant thesis sentence and supporting claim-bearing sentences.',
                'strict': True,
                'schema': CLAIM_CODING_SCHEMA,
            },
        },
    )

    output_text = extract_output_text(response)
    if not output_text:
        raise ValueError('The API response did not include any output_text to parse.')

    payload = json.loads(output_text)
    return validate_claim_coding_output(payload, sentence_df, response=response)


def result_sentence_table(coding_result, sentence_df):
    lookup = sentence_lookup(sentence_df)
    rows = [
        {
            'role': 'thesis',
            'sentence_id': coding_result['thesis_sentence_id'],
            'sentence': lookup[coding_result['thesis_sentence_id']],
        }
    ]

    for idx, sentence_id in enumerate(coding_result['support_sentence_ids'], start=1):
        rows.append(
            {
                'role': f'support_{idx}',
                'sentence_id': sentence_id,
                'sentence': lookup[sentence_id],
            }
        )

    for idx, sentence_id in enumerate(coding_result['secondary_claim_ids'], start=1):
        rows.append(
            {
                'role': f'candidate_{idx}',
                'sentence_id': sentence_id,
                'sentence': lookup[sentence_id],
            }
        )

    return pd.DataFrame(rows)


def flatten_claim_coding_result(article_row, coding_result, sentence_df):
    lookup = sentence_lookup(sentence_df)
    return {
        'article_id': article_row.get('id', None),
        'title': article_row.get('title', None),
        'central_claim_summary': coding_result['central_claim_summary'],
        'has_clear_central_thesis': coding_result['has_clear_central_thesis'],
        'thesis_sentence_id': coding_result['thesis_sentence_id'],
        'thesis_sentence': lookup[coding_result['thesis_sentence_id']],
        'support_sentence_ids': coding_result['support_sentence_ids'],
        'support_sentences': [lookup[sid] for sid in coding_result['support_sentence_ids']],
        'secondary_claim_ids': coding_result['secondary_claim_ids'],
        'secondary_claim_sentences': [lookup[sid] for sid in coding_result['secondary_claim_ids']],
        'response_id': coding_result.get('response_id'),
        'model': coding_result.get('model'),
        'usage_input_tokens': coding_result.get('usage', {}).get('input_tokens') if coding_result.get('usage') else None,
        'usage_output_tokens': coding_result.get('usage', {}).get('output_tokens') if coding_result.get('usage') else None,
        'usage_total_tokens': coding_result.get('usage', {}).get('total_tokens') if coding_result.get('usage') else None,
        'usage_reasoning_tokens': coding_result.get('usage', {}).get('reasoning_tokens') if coding_result.get('usage') else None,
    }


def code_articles_dataframe(
    articles_df,
    title_col='title',
    text_col='body_text',
    max_articles=3,
    model=MODEL,
    reasoning_effort=REASONING_EFFORT,
    max_output_tokens=MAX_OUTPUT_TOKENS,
    sleep_seconds=0.0,
):
    rows = []

    for _, article_row in articles_df.head(max_articles).iterrows():
        try:
            sentence_df = sentence_table_from_text(article_row[text_col])
            coding_result = call_claim_coder(
                sentence_df=sentence_df,
                article_title=article_row.get(title_col),
                model=model,
                reasoning_effort=reasoning_effort,
                max_output_tokens=max_output_tokens,
            )
            row_payload = flatten_claim_coding_result(article_row, coding_result, sentence_df)
            row_payload['error'] = None
            rows.append(row_payload)
        except Exception as exc:
            rows.append(
                {
                    'article_id': article_row.get('id', None),
                    'title': article_row.get(title_col, None),
                    'central_claim_summary': None,
                    'has_clear_central_thesis': None,
                    'thesis_sentence_id': None,
                    'support_sentence_ids': None,
                    'secondary_claim_ids': None,
                    'usage_total_tokens': None,
                    'error': f'{type(exc).__name__}: {exc}',
                }
            )

        if sleep_seconds:
            time.sleep(sleep_seconds)

    return pd.DataFrame(rows)


In [12]:
RAW_DATA_DIR = PROJECT_ROOT / 'backend' / 'data' / 'raw' / 'guardian_by_year'
YEARS = [2024]
MIN_BODY_TEXT_CHARS = 1000

try:
    articles = load_and_clean_guardian_years(
        years=YEARS,
        folder=RAW_DATA_DIR,
        min_body_text_chars=MIN_BODY_TEXT_CHARS,
    )
    print(f"Loaded {len(articles):,} cleaned Guardian opinion articles from {YEARS}.")
    display(articles[['id', 'date', 'title', 'author_display', 'year']].head(10))
except Exception as exc:
    articles = pd.DataFrame()
    print(f"Local article load skipped: {type(exc).__name__}: {exc}")
    print('Set MANUAL_ARTICLE_TEXT in the config cell if you want to work without the local Guardian dataset.')


Loaded 4,856 cleaned Guardian opinion articles from [2024].


,id,date,title,author_display,year
0,commentisfree/2024/jan/01/failed-politics-workplace-strikes-gaza-war-protests,2024-01-01 06:00:23+00:00,"After a year of failed politics, we know we can’t rely on leaders. Luckily, we have ourselves",Nesrine Malik,2024
1,commentisfree/2024/jan/01/princess-mary-queen-margrethe-ii-abdication-frederik-relationship-timeline,2024-01-01 06:05:23+00:00,The promotion of Australian-born Mary from princess to queen proves what a pure lottery the aristocracy has always been,Van Badham,2024
2,commentisfree/2024/jan/01/grandmother-walnut-tree-fire-flood-slovenia-recipe,2024-01-01 07:00:25+00:00,My grandmother’s walnut tree didn’t survive fires and floods – but she left us a recipe for hope,Ana Schnabl,2024
3,commentisfree/2024/jan/01/ravish-kumar-journalism-democracy-threat-lies,2024-01-01 08:00:26+00:00,"In a dark world, a light is held by those who make it harder for the powerful to lie",Zoe Williams,2024
4,commentisfree/2024/jan/01/channel-tunnel-uk-european-30th-anniversary,2024-01-01 10:00:28+00:00,"The Channel tunnel should have made the UK truly European, but didn’t. We must get back on track",Christian Wolmar,2024
5,commentisfree/2024/jan/01/nikki-haleys-comment-on-the-us-civil-war-was-no-gaffe,2024-01-01 11:14:29+00:00,Nikki Haley’s comment on the US civil war was no gaffe,Sidney Blumenthal,2024
6,commentisfree/2024/jan/01/jails-too-soft-hardened-criminals-community-sentences,2024-01-01 12:00:29+00:00,"If you think jails are too soft and full of hardened criminals, read this and think again",Alex South,2024
7,commentisfree/2024/jan/02/ghosts-are-suddenly-in-vogue-perhaps-we-believe-in-them-more-than-we-care-to-admit,2024-01-01 15:00:34+00:00,Ghosts are suddenly in vogue. Perhaps we believe in them more than we care to admit,Elfy Scott,2024
8,commentisfree/2024/jan/01/rishi-sunak-boris-johnson-david-cameron-liz-truss-dominic-cummings-politics,2024-01-01 15:00:34+00:00,"Sunak, Johnson, Cameron, Truss – and Cummings in the frame again. This inept elite is playing us all for fools",Hugh Muir,2024
9,commentisfree/2024/jan/01/the-guardian-view-on-digital-only-archives-material-items-still-matter-to-historians,2024-01-01 18:00:37+00:00,The Guardian view on digital-only archives: material items still matter to historians,Editorial,2024


In [30]:
MANUAL_ARTICLE_TEXT = None
MANUAL_ARTICLE_TITLE = 'Manual article'

ARTICLE_ID = None
TITLE_QUERY = 'israel'
ARTICLE_INDEX = 0
SHOW_SENTENCE_ROWS = 40

MODEL = 'gpt-5-nano'
REASONING_EFFORT = 'minimal'
MAX_OUTPUT_TOKENS = 250


In [31]:
matching_articles, selected_article = select_article(
    articles=articles,
    article_id=ARTICLE_ID,
    title_query=TITLE_QUERY,
    article_index=ARTICLE_INDEX,
    manual_text=MANUAL_ARTICLE_TEXT,
    manual_title=MANUAL_ARTICLE_TITLE,
)

display(matching_articles[['id', 'date', 'title', 'author_display']].head(10))
preview_article(selected_article)

sentence_df = sentence_table_from_text(selected_article['body_text'])
print(f"Prepared {len(sentence_df):,} sentence IDs for prompting.")
sentence_df.head(SHOW_SENTENCE_ROWS)


,id,date,title,author_display
0,commentisfree/2024/jan/02/israel-allies-peace-talks-include-hamas-palestinians,2024-01-02 15:00:35+00:00,"Israel and its allies must face facts: peace talks are the only way forward, and they will have to include Hamas",Peter Hain
1,commentisfree/2024/jan/03/vivian-silver-peace-activism-israel-gaza-conflict,2024-01-03 08:00:16+00:00,"Out of the ashes, the dignity and compassion of Israeli peace activists gives me hope",Clive Myrie
2,commentisfree/2024/jan/05/israel-hezbollah-war-hamas-saleh-arouri-lebanon,2024-01-05 14:54:53+00:00,Israel is pushing Hezbollah to its limits. How it responds will define the future of this war,Amal Saad
3,commentisfree/2024/jan/08/israel-gaza-war-wider-conflict-us-iran,2024-01-08 11:00:39+00:00,"Could the Israel-Gaza war spark a wider conflict involving the US, Iran or others?","Rajan Menon, Daniel R DePetris"
4,commentisfree/2024/jan/09/israel-media-gaza-benjamin-netanyahu-settler-movement,2024-01-09 12:00:09+00:00,The far right infiltration of Israel’s media is blinding the public to the truth about Gaza,Etan Nechin
5,commentisfree/2024/jan/10/israel-murdering-palestinian-journalists-in-gaza,2024-01-10 08:00:25+00:00,Israel is murdering Palestinian journalists in Gaza. Where is the outrage?,Chris McGreal
6,commentisfree/2024/jan/10/israel-palestinians-genocide-south-africa-apartheid,2024-01-10 15:36:57+00:00,The ghost of apartheid has come back to haunt Israel and give hope to Palestinians,Tony Karon
7,commentisfree/2024/jan/11/israeli-women-and-girls-have-suffered-horrific-sexual-violence-from-hamas-where-is-the-outrage,2024-01-11 10:00:38+00:00,Israeli women and girls have suffered horrific sexual violence from Hamas. Where is the outrage?,"Deborah Lipstadt, Michèle Taylor"
8,commentisfree/2024/jan/11/israel-aid-gaza-britain-crossings-water-united-nations-food,2024-01-11 18:30:07+00:00,Israel must act now to let aid through and save lives in Gaza. Britain has a plan to help that happen,David Cameron
9,commentisfree/2024/jan/11/the-guardian-view-on-israel-and-allegations-of-genocide-a-case-that-needs-to-be-heard,2024-01-11 18:46:07+00:00,The Guardian view on Israel and allegations of genocide: a case that needs to be heard,Editorial


## Israel and its allies must face facts: peace talks are the only way forward, and they will have to include Hamas

id: commentisfree/2024/jan/02/israel-allies-peace-talks-include-hamas-palestinians
date: 2024-01-02
author(s): Peter Hain

Body preview:
After the Hamas terror of 7 October and Benjamin Netanyahu’s horrific retaliation in Gaza, some long overdue truths need stating. First, Israel is not going to “destroy Hamas”, as its leaders promise – not even by destroying Gaza. Although Israel is damaging Hamas militarily, maybe significantly, with many of its tunnels eliminated and its fighters fleeing, Hamas is a movement and an ideology that, in many respects, Netanyahu’s extremism helped to promote. Rightwing Israeli governments have thwarted serious negotiations with Palestine’s more “moderate“ party, the late Yasser Arafat’s Fatah, since the Camp David summit in 2000 – more than 20 years ago. They have also consistently oppressed Gaza residents, imposing a near-constant state of siege. Is it really surprising that many Palestinians turned in desperation to an extremist alternative in Hamas? T

,sentence_id,sentence
0,s0,"After the Hamas terror of 7 October and Benjamin Netanyahu’s horrific retaliation in Gaza, some long overdue truths need stating."
1,s1,"First, Israel is not going to “destroy Hamas”, as its leaders promise – not even by destroying Gaza."
2,s2,"Although Israel is damaging Hamas militarily, maybe significantly, with many of its tunnels eliminated and its fighters fleeing, Hamas is a movement and an ideology that, in ma..."
3,s3,"Rightwing Israeli governments have thwarted serious negotiations with Palestine’s more “moderate“ party, the late Yasser Arafat’s Fatah, since the Camp David summit in 2000 – m..."
4,s4,"They have also consistently oppressed Gaza residents, imposing a near-constant state of siege."
5,s5,Is it really surprising that many Palestinians turned in desperation to an extremist alternative in Hamas?
6,s6,The lesson of all modern conflicts must be that failure by the powerful to end injustice and negotiate a solution breeds extremism.
7,s7,"As Britain’s troubled history in Northern Ireland vividly demonstrates, when politics doesn’t work, violence fills the vacuum."
8,s8,British governments refused for decades to officially negotiate with the IRA because of its terrorist outrages.
9,s9,"But when they finally did so, it resulted in the 1998 Good Friday agreement."


In [32]:
coding_result = call_claim_coder(
    sentence_df=sentence_df,
    article_title=selected_article.get('title'),
    model=MODEL,
    reasoning_effort=REASONING_EFFORT,
    max_output_tokens=MAX_OUTPUT_TOKENS,
)

print(json.dumps(coding_result, indent=2))


{
  "central_claim_summary": "Peace negotiations, including Hamas, are the only viable path forward in the Israeli\u2013Palestinian conflict and must involve broader regional diplomacy.",
  "has_clear_central_thesis": true,
  "thesis_sentence_id": "s13",
  "support_sentence_ids": [
    "s6",
    "s14"
  ],
  "secondary_claim_ids": [
    "s18",
    "s27",
    "s28"
  ],
  "response_id": "resp_07917bcd4d8955a80169bca8074c0881959f7ad65266800048",
  "model": "gpt-5-nano-2025-08-07",
  "usage": {
    "input_tokens": 2261,
    "output_tokens": 98,
    "total_tokens": 2359,
    "reasoning_tokens": 0
  }
}


In [28]:
display(Markdown(f"**Central claim summary:** {coding_result['central_claim_summary']}"))
display(Markdown(f"**Has clear central thesis:** {coding_result['has_clear_central_thesis']}"))
result_sentence_table(coding_result, sentence_df)


**Central claim summary:** A Green Cross for climate disasters should be created as a planetary civil protection force to coordinate, fund, and standardize international relief, analogous to the Red Cross.

**Has clear central thesis:** True

,role,sentence_id,sentence
0,thesis,s15,"And so here is a complementary idea: if there is a Red Cross, founded to support people affected by armed violence, why can we not found a Green Cross for those affected by cli..."
1,support_1,s19,Imagine instead a planetary civil protection force with rapid-reaction capacities and clear operational expertise guaranteed by regular joint training and funding.
2,support_2,s22,"Just like the International Committee of the Red Cross, this body could be independent of any one country but draw its funding from both public and private sectors, working wit..."
3,candidate_1,s31,A planetary civil protection body could eventually expand from climate disaster relief to the preservation of nature.
4,candidate_2,s34,"Ultimately, a global initiative of this kind would do more than materially protect the world’s inhabitants: it would foster a much-needed feeling of common belonging to a share..."
5,candidate_3,s37,A civil protection force for the planet would represent a limited but concrete place to start.


In [34]:
BATCH_ARTICLE_LIMIT = 3
BATCH_SLEEP_SECONDS = 0.0

batch_results = code_articles_dataframe(
    articles_df=articles,
    max_articles=BATCH_ARTICLE_LIMIT,
    model=MODEL,
    reasoning_effort=REASONING_EFFORT,
    max_output_tokens=MAX_OUTPUT_TOKENS,
    sleep_seconds=BATCH_SLEEP_SECONDS,
)

batch_results[[
    'article_id',
    'title',
    'central_claim_summary',
    'has_clear_central_thesis',
    'thesis_sentence_id',
    'support_sentence_ids',
    'secondary_claim_ids',
    'usage_total_tokens',
    'error',
]]


,article_id,title,central_claim_summary,has_clear_central_thesis,thesis_sentence_id,support_sentence_ids,secondary_claim_ids,usage_total_tokens,error
0,commentisfree/2024/jan/01/failed-politics-workplace-strikes-gaza-war-protests,"After a year of failed politics, we know we can’t rely on leaders. Luckily, we have ourselves","There is still a possibility to reclaim agency and effect change through people’s own actions and collective democratic engagement, even as politics remains divided and leaders...",True,s40,"[s22, s23]","[s31, s32, s35]",2672,None
1,commentisfree/2024/jan/01/princess-mary-queen-margrethe-ii-abdication-frederik-relationship-timeline,The promotion of Australian-born Mary from princess to queen proves what a pure lottery the aristocracy has always been,The promotion of Mary from princess to queen reveals that aristocracy is essentially a pure lottery perpetuated by hereditary privilege rather than merit or modern democratic l...,True,s15,"[s5, s10]","[s23, s25]",2229,None
2,commentisfree/2024/jan/01/grandmother-walnut-tree-fire-flood-slovenia-recipe,My grandmother’s walnut tree didn’t survive fires and floods – but she left us a recipe for hope,"Culture should be safeguarded and adapted to fit the times, even as it faces decay and environmental destruction.",True,s7,"[s6, s39]","[s42, s44, s48]",2270,None
